# Scratch seq2seq — inference runs only

Greedy-decode headings from **`best.pt`** checkpoints and write **one JSON per model size** under **`artifacts/inference_runs/`** (`{dev|test}_scratch_{tiny|medium}.json`).

**Workflow:** run this notebook when checkpoints or eval settings change. Then use **[`03_automated_metrics.ipynb`](03_automated_metrics.ipynb)** for ROUGE/F1/chrF and **[`02_llm_judge.ipynb`](02_llm_judge.ipynb)** for LLM judge scores — those notebooks **only import** these files (they do not load PyTorch checkpoints).

**Requirements:** `HUGGINGFACE_HUB_TOKEN` (or `HF_TOKEN`) in `.env`, matching **`EVAL_SPLIT`** / **`EVAL_N`** below with what you want downstream notebooks to consume.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src" / "briefme").is_dir():
        return cwd
    if (cwd.parent / "src" / "briefme").is_dir():
        return cwd.parent
    return cwd


REPO_ROOT = _repo_root()
SRC_ROOT = REPO_ROOT / "src"

try:
    import briefme as _briefme_check  # noqa: F401
except ImportError:
    if SRC_ROOT.is_dir():
        sys.path.insert(0, str(SRC_ROOT))

load_dotenv(REPO_ROOT / ".env", override=True)

from briefme.data import SPLIT_DEV, SPLIT_TEST, load_arg_summ_split_streaming, materialize_head
from briefme.inference_persist import inference_runs_dir, save_scratch_inference_json
from briefme.metrics import aggregate
from briefme.schema import to_seq2seq_example
from briefme.scratch_inference import greedy_predict_headings, load_scratch_seq2seq_checkpoint

print("Repo:", REPO_ROOT)

In [ ]:
# --- knobs (keep EVAL_SPLIT / EVAL_N aligned with 02 / 03) ---
EVAL_SPLIT = SPLIT_DEV  # or SPLIT_TEST
EVAL_N = 30
PRED_BATCH = 8
INF_DEVICE = "cpu"  # "cuda" / "mps" / "cpu"

CKPT_TINY = REPO_ROOT / "runs" / "notebook_scratch_tiny_full" / "best.pt"
CKPT_MEDIUM = REPO_ROOT / "runs" / "notebook_scratch_medium_full" / "best.pt"

split_tag = "dev" if EVAL_SPLIT == SPLIT_DEV else "test"
INFERENCE_DIR = inference_runs_dir(REPO_ROOT)

if not (os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN")):
    raise RuntimeError(
        "Set HUGGINGFACE_HUB_TOKEN (or HF_TOKEN) in .env to load splits from the Hub."
    )

eval_ds = materialize_head(load_arg_summ_split_streaming(EVAL_SPLIT), EVAL_N)
examples = [to_seq2seq_example(row) for row in eval_ds]
refs = [ex["target"] for ex in examples]

for label, ckpt_path in [("tiny", CKPT_TINY), ("medium", CKPT_MEDIUM)]:
    if not ckpt_path.is_file():
        print(f"[skip] no checkpoint at {ckpt_path}")
        continue

    print(f"\n=== {split_tag} · scratch {label} ({ckpt_path}) ===")
    model, tokenizer, tcfg, src_prefix = load_scratch_seq2seq_checkpoint(ckpt_path, device=INF_DEVICE)
    preds = greedy_predict_headings(
        model,
        tokenizer,
        examples,
        tcfg=tcfg,
        source_prefix=src_prefix,
        batch_size=PRED_BATCH,
    )
    agg = aggregate(preds, refs)
    save_path = INFERENCE_DIR / f"{split_tag}_scratch_{label}.json"
    save_scratch_inference_json(
        save_path,
        split_tag=split_tag,
        eval_n=EVAL_N,
        label=label,
        checkpoint=ckpt_path,
        preds=preds,
        refs=refs,
        sources=[ex["source"] for ex in examples],
        agg=agg,
    )
    print(f"Saved → {save_path.relative_to(REPO_ROOT)}")
    print(
        "corpus:",
        {k: round(float(agg[k]), 4) for k in ("rouge1_f", "rougeL_f", "token_f1_macro", "chrf_corpus")},
    )
    del model

### Saved runs (optional)

Lists JSON files under **`artifacts/inference_runs/`** (newest first).

In [ ]:
from briefme.inference_persist import list_saved_scratch_runs

paths = list_saved_scratch_runs(REPO_ROOT)
for p in paths[:20]:
    print(p.relative_to(REPO_ROOT))